# Day 28：文档加载与分割(RAG第一步)：多格式解析器与中文递归分块调优（Jupyter Notebook 实战手册）

> **所属阶段**：第 3 阶段 · 第三阶段：LangChain 与企业级 RAG 开发 ｜ **第 4 周**：第4周：LangChain 核心架构与 RAG 文档预处理  
> **GitLab 工单**：`STARMIND-028` ｜ **开发分支**：`feat/day28-rag-loaders-chinese-splitter`  
> **配套 HTML5 课件**：`courseware/day28/index.html` ｜ **独立版本快照**：`versions/day28/`

---

## 🎙️ 老陈开场旁白

各位早！今天正式推开企业级 RAG（检索增强生成）的大门！老陈做了十几年搜索和 AI，送大家一句血泪总结：**RAG 系统 70% 的线上 Bad Case，根本不是大模型笨，而是第一步「文档解析与切分」没做好！** 比如把一张报销标准表拦腰斩断，前半段在 Chunk 1，金额数字在 Chunk 2；或者拿英文默认的 `['\n\n', '\n', ' ', '']` 分隔符去切中文文档，中文压根不用空格分词，结果直接一刀把句子从中间劈开！今天咱们从零实现统一 `Document` 模型、多格式加载器（MD/HTML/CSV/PDF/Word）以及针对中文深度优化的 `RecursiveCharacterTextSplitter`，最后跑一组 `chunk_size` 和 `chunk_overlap` 调优实验，用数据说话！

### 承上启下
- **昨日基线**：昨天（Day 27）我们搞定了多轮会话记忆与 SQLite 持久化，星辰智脑已经能记住员工上下文了。但现在它还是个「空壳子」——公司内网散落着几千份 Markdown 技术文档、HTML 公告、CSV 薪资/资产表、PDF 员工手册和 Word 制度文件，大模型压根没读过。
- **明日预告**：明天（Day 29）我们将把今天切好的高质量文本块（Chunks）向量化，深入 HNSW 近似最近邻检索原理，亲手搭建支持元数据过滤的本地持久化向量数据库！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day28'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-028）

杭州星辰数科知识库首批需要接入 5 类企业异构文档：研发 Markdown 规范、内网 HTML 公告、IT 资产/排班 CSV 表格、HR PDF 员工手册与法务 Word 合同模板。

**今日迭代目标：**
- 定义统一的 `Document(page_content, metadata)` 数据结构，与 LangChain `langchain_core.documents.Document` 对齐。
- 实现 Markdown、HTML、CSV、纯文本及结构化 PDF/Word 加载器，自动清洗噪音并提取丰富元数据。
- 实现中文增强版 `RecursiveCharacterTextSplitter`，并开展不同 `chunk_size` / `chunk_overlap` 参数组合的断句完整度与冗余率实验。

**验收标准：**
- HTML 加载器能彻底剥离 `<script>`/`<style>` 与 HTML 标签，仅保留干净正文；Markdown 加载器能提取 `#` 标题层级。
- 中文递归分块器严格遵守 `len(chunk) <= chunk_size`（除非单个原子片段超长），且相邻分块正确保留 overlap 上下文。
- 分块调优实验能输出量化对比表，所有 pytest 单元测试 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 中文文档分块为什么不能直接用 LangChain 默认分隔符？

LangChain 默认 `RecursiveCharacterTextSplitter()` 的 `separators` 是 `['\n\n', '\n', ' ', '']`。英文单词之间有空格，所以第三级 `' '` 能按词切；但中文段落里没有空格，一旦某个自然段超过 `chunk_size`，默认配置就会直接跳到 `''`（按单字硬切），把一句话拦腰劈成两半！在国内项目中，务必显式传入中文标点列表：`['\n\n', '\n', '。', '！', '？', '；', '，', ' ', '']`。

> **⚠️ 老陈避坑**：踩坑提醒：切完以后记得把标点符号保留在上一句末尾（`keep_separator='end'`），不要把句号留到下一块的开头。

### 2. 2. 生产环境 LangChain 多格式 Loader 对照

在实际企业项目中，针对扫描件 PDF 和复杂表格，通常组合使用以下生态组件：
```python
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, CSVLoader, BSHTMLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=['\n\n', '\n', '。', '！', '？', '；', '，', ' ', ''],
)
chunks = splitter.split_documents(docs)
```
如果是含大量跨页表格/双栏排版的复杂 PDF，国内大厂强烈推荐开源神器 **MinerU** 或 **Docling** 先把 PDF 转成高质量 Markdown，再喂给 RAG 流水线，效果比 `PyPDFLoader` 提升一个量级！

> **⚠️ 老陈避坑**：踩坑提醒：CSV/Excel 表格绝对不要当成纯文本一坨切分！必须把每一行连同「表头列名」一起拼成自然语言（例如 `城市: 杭州 | 住宿上限: 500元 | 餐补: 100元`），否则切片丢了表头，大模型根本不知道 `500` 是什么意思。

### 3. 3. chunk_size 到底设多少合适？

没有放之四海而皆准的魔法数字，取决于你的文档类型和 Embedding 模型：
- **FAQ / 规章条款类**（如 HR 制度、报销标准）：推荐 `250 ~ 400` 字，overlap `40 ~ 60` 字，检索粒度精准。
- **长篇技术方案 / 架构文档**：推荐 `500 ~ 800` 字，overlap `80 ~ 120` 字，避免代码片段或上下文被切碎。
- 最佳工程方法：像我们今天下午做的一样，用你的真实业务问题跑一组消融实验，看检索命中率定参数！

> **⚠️ 老陈避坑**：踩坑提醒：`chunk_overlap` 不宜超过 `chunk_size` 的 25%，否则向量库体积和检索去重成本会明显上升，甚至 Top-3 全是高度重复的相邻切片。


## 💻 实操步骤 1：步骤一：实现统一 Document 模型与多格式企业文档加载器

- **目标文件**：`versions/day28/src/day28_loaders.py`
- **老陈代码旁白**：咱们先统一数据结构 `Document`，然后一口气实现 5 种企业最常见格式的加载器：Markdown（按标题提取章节元数据）、HTML（基于标准库 `HTMLParser` 彻底剔除 script/style 噪音）、CSV（每行带表头语义化）、真实 `.docx`（标准库 zipfile 解析 Word XML）以及模拟分页 PDF 加载器。


In [ ]:
"""
Day 28 - 步骤一：统一 Document 模型与多格式文档加载器（MD/HTML/CSV/DOCX/PDF）
对应生产包：langchain_core.documents / langchain_community.document_loaders
"""

from __future__ import annotations

import csv
import io
import re
import xml.etree.ElementTree as ET
import zipfile
from dataclasses import dataclass, field
from html.parser import HTMLParser
from pathlib import Path
from typing import Any, Dict, List


@dataclass
class Document:
    """统一文档片段模型，100% 对齐 langchain_core.documents.Document。"""

    page_content: str
    metadata: Dict[str, Any] = field(default_factory=dict)


class _CleanHTMLParser(HTMLParser):
    """过滤 <script>/<style>/<nav> 标签，仅提取干净正文。"""

    SKIP_TAGS = {"script", "style", "noscript", "nav", "footer"}

    def __init__(self) -> None:
        super().__init__()
        self._skip_depth = 0
        self._parts: List[str] = []

    def handle_starttag(self, tag: str, attrs: Any) -> None:
        if tag.lower() in self.SKIP_TAGS:
            self._skip_depth += 1
        elif tag.lower() in {"p", "div", "br", "li", "h1", "h2", "h3", "tr"} and self._skip_depth == 0:
            self._parts.append("\n")

    def handle_endtag(self, tag: str) -> None:
        if tag.lower() in self.SKIP_TAGS and self._skip_depth > 0:
            self._skip_depth -= 1

    def handle_data(self, data: str) -> None:
        if self._skip_depth == 0:
            text = data.strip()
            if text:
                self._parts.append(text)

    def get_text(self) -> str:
        raw = " ".join(self._parts)
        raw = re.sub(r"[ \t]*\n[ \t]*", "\n", raw)
        return re.sub(r"\n{3,}", "\n\n", raw).strip()


class MarkdownLoader:
    """按一级/二级标题切分 Markdown，并把章节标题写入 metadata。"""

    def load_text(self, text: str, source: str = "inline.md") -> List[Document]:
        docs: List[Document] = []
        current_title = "概述"
        buffer: List[str] = []

        def flush() -> None:
            content = "\n".join(buffer).strip()
            if content:
                docs.append(
                    Document(
                        page_content=content,
                        metadata={"source": source, "doc_type": "markdown", "section": current_title},
                    )
                )

        for line in text.splitlines():
            heading = re.match(r"^(#{1,3})\s+(.+)$", line.strip())
            if heading:
                flush()
                buffer = []
                current_title = heading.group(2).strip()
            else:
                buffer.append(line)
        flush()
        return docs

    def load(self, file_path: str | Path) -> List[Document]:
        path = Path(file_path)
        return self.load_text(path.read_text(encoding="utf-8"), source=path.name)


class HTMLLoader:
    """加载 HTML 并剥离脚本、样式与导航噪音。"""

    def load_text(self, html: str, source: str = "inline.html") -> List[Document]:
        parser = _CleanHTMLParser()
        parser.feed(html)
        clean_text = parser.get_text()
        return [Document(page_content=clean_text, metadata={"source": source, "doc_type": "html"})]

    def load(self, file_path: str | Path) -> List[Document]:
        path = Path(file_path)
        return self.load_text(path.read_text(encoding="utf-8"), source=path.name)


class CSVLoader:
    """将 CSV 每一行与表头拼接为语义完整的自然语言 Document，避免表头丢失。"""

    def load_text(self, csv_text: str, source: str = "inline.csv") -> List[Document]:
        reader = csv.DictReader(io.StringIO(csv_text.strip()))
        docs: List[Document] = []
        for row_idx, row in enumerate(reader, start=1):
            content = " | ".join(f"{k.strip()}: {(v or '').strip()}" for k, v in row.items() if k)
            docs.append(
                Document(
                    page_content=content,
                    metadata={"source": source, "doc_type": "csv", "row": row_idx},
                )
            )
        return docs


class DocxLoader:
    """零第三方依赖解析真实 .docx 文件（读取 ZIP 内 word/document.xml 段落）。"""

    W_NS = "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}"

    def load(self, file_path: str | Path) -> List[Document]:
        path = Path(file_path)
        paragraphs: List[str] = []
        with zipfile.ZipFile(path, "r") as zf:
            xml_bytes = zf.read("word/document.xml")
        root = ET.fromstring(xml_bytes)
        for p in root.iter(f"{self.W_NS}p"):
            texts = [t.text for t in p.iter(f"{self.W_NS}t") if t.text]
            joined = "".join(texts).strip()
            if joined:
                paragraphs.append(joined)
        return [
            Document(
                page_content="\n".join(paragraphs),
                metadata={"source": path.name, "doc_type": "docx", "paragraphs": len(paragraphs)},
            )
        ]


class PagePDFLoader:
    """
    分页 PDF 加载器（离线教学版按分页符 `\\f` 或 `---PAGE---` 模拟真实 PyPDFLoader 分页输出；
    生产环境直接替换为 langchain_community.document_loaders.PyPDFLoader 或 MinerU）。
    """

    def load_pages(self, raw_pages: List[str], source: str = "handbook.pdf") -> List[Document]:
        docs: List[Document] = []
        for page_num, content in enumerate(raw_pages, start=1):
            cleaned = re.sub(r"\n{3,}", "\n\n", content.strip())
            if cleaned:
                docs.append(
                    Document(
                        page_content=cleaned,
                        metadata={"source": source, "doc_type": "pdf", "page": page_num},
                    )
                )
        return docs


if __name__ == "__main__":
    md_docs = MarkdownLoader().load_text("# 报销制度\n差旅住宿一线城市上限600元。\n## 审批流程\n需直属主管审批。")
    html_docs = HTMLLoader().load_text("<html><script>alert(1)</script><body><h1>公告</h1><p>周五晚8点升级VPN。</p></body></html>")
    csv_docs = CSVLoader().load_text("城市,住宿上限,餐补\n杭州,500元,100元\n上海,600元,120元")
    pdf_docs = PagePDFLoader().load_pages(["第1页：员工入职满1年享5天年假。", "第2页：年假需提前3个工作日申请。"])

    for d in [*md_docs, *html_docs, *csv_docs, *pdf_docs]:
        print(f"[{d.metadata['doc_type']}] {d.metadata} -> {d.page_content}")


### 🔍 步骤 1 代码深度解析

1. `_CleanHTMLParser` 维护 `_skip_depth` 计数器，彻底过滤 `<script>`/`<style>`/`<nav>` 里的所有垃圾文本，同时在块级标签（`<p>`/`<div>`/`<li>`）自动插入换行保留段落结构。
2. `DocxLoader` 利用 `.docx` 本质是 OpenXML ZIP 包的特性，用 15 行标准库代码就能真枪实弹解析 Word 文档，在极简容器镜像里非常实用。
3. `CSVLoader` 把每一行转成 `城市: 杭州 | 住宿上限: 500元 | 餐补: 100元`，让向量检索能精准命中表头与数值的对应关系。

**预期运行输出参考：**
```text
[markdown] {'source': 'inline.md', 'doc_type': 'markdown', 'section': '报销制度'} -> 差旅住宿一线城市上限600元。
[markdown] {'source': 'inline.md', 'doc_type': 'markdown', 'section': '审批流程'} -> 需直属主管审批。
[html] {'source': 'inline.html', 'doc_type': 'html'} -> 公告
周五晚8点升级VPN。
[csv] {'source': 'inline.csv', 'doc_type': 'csv', 'row': 1} -> 城市: 杭州 | 住宿上限: 500元 | 餐补: 100元
[csv] {'source': 'inline.csv', 'doc_type': 'csv', 'row': 2} -> 城市: 上海 | 住宿上限: 600元 | 餐补: 120元
[pdf] {'source': 'handbook.pdf', 'doc_type': 'pdf', 'page': 1} -> 第1页：员工入职满1年享5天年假。
[pdf] {'source': 'handbook.pdf', 'doc_type': 'pdf', 'page': 2} -> 第2页：年假需提前3个工作日申请。
```


## 💻 实操步骤 2：步骤二：实现中文 RecursiveCharacterTextSplitter 与分块参数调优实验

- **目标文件**：`versions/day28/src/day28_text_splitter.py`
- **老陈代码旁白**：文档加载进来了，接下来亲手写中文增强版 `RecursiveCharacterTextSplitter`，并跑一组 `chunk_size` 与 `chunk_overlap` 的对比实验，直观量化不同参数下的切片数量、完整句占比与总字符膨胀率！


In [ ]:
"""
Day 28 - 步骤二：中文递归分块器（RecursiveCharacterTextSplitter）与参数调优实验
对应生产包：langchain_text_splitters.RecursiveCharacterTextSplitter
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import Any, Dict, List, Sequence

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document


class RecursiveCharacterTextSplitter:
    """
    中文增强版递归字符文本分割器：
    按语义优先级（段落 -> 换行 -> 句末标点 -> 逗号 -> 字符）递归切分，
    并通过 chunk_overlap 保持相邻切片的上下文连贯性。
    """

    DEFAULT_ZH_SEPARATORS = ["\n\n", "\n", "。", "！", "？", "；", "，", " ", ""]

    def __init__(
        self,
        chunk_size: int = 200,
        chunk_overlap: int = 30,
        separators: Sequence[str] | None = None,
    ) -> None:
        if chunk_size <= 0:
            raise ValueError("chunk_size 必须大于 0")
        if chunk_overlap < 0 or chunk_overlap >= chunk_size:
            raise ValueError("chunk_overlap 必须满足 0 <= chunk_overlap < chunk_size")
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.separators = list(separators) if separators is not None else list(self.DEFAULT_ZH_SEPARATORS)

    def _split_with_separator(self, text: str, sep: str) -> List[str]:
        """按分隔符切分，并将标点分隔符保留在前一段末尾，保护中文句末标点。"""
        if sep == "":
            return list(text)
        raw_parts = text.split(sep)
        pieces: List[str] = []
        for idx, part in enumerate(raw_parts):
            if idx < len(raw_parts) - 1:
                pieces.append(part + sep)
            elif part:
                pieces.append(part)
        return [p for p in pieces if p]

    def _recursive_split(self, text: str, separators: List[str]) -> List[str]:
        if len(text) <= self.chunk_size:
            return [text.strip()] if text.strip() else []

        # 找到第一个在当前文本中出现的分隔符
        chosen_sep = ""
        remaining_seps: List[str] = []
        for i, sep in enumerate(separators):
            if sep == "" or sep in text:
                chosen_sep = sep
                remaining_seps = separators[i + 1 :]
                break

        splits = self._split_with_separator(text, chosen_sep)
        final_chunks: List[str] = []
        good_splits: List[str] = []

        for piece in splits:
            if len(piece) <= self.chunk_size:
                good_splits.append(piece)
            else:
                if good_splits:
                    final_chunks.extend(self._merge_splits(good_splits))
                    good_splits = []
                if remaining_seps:
                    final_chunks.extend(self._recursive_split(piece, remaining_seps))
                else:
                    # 最终定长硬切兜底
                    step = self.chunk_size - self.chunk_overlap
                    for start in range(0, len(piece), step):
                        sub = piece[start : start + self.chunk_size].strip()
                        if sub:
                            final_chunks.append(sub)
        if good_splits:
            final_chunks.extend(self._merge_splits(good_splits))
        return final_chunks

    def _merge_splits(self, splits: List[str]) -> List[str]:
        """将细粒度片段合并为接近 chunk_size 的块，并保留 chunk_overlap 重叠尾部。"""
        chunks: List[str] = []
        current: List[str] = []
        current_len = 0

        for piece in splits:
            piece_len = len(piece)
            if current and current_len + piece_len > self.chunk_size:
                chunk_text = "".join(current).strip()
                if chunk_text:
                    chunks.append(chunk_text)
                # 从末尾回溯保留不超过 chunk_overlap 的上下文片段
                overlap_buf: List[str] = []
                overlap_len = 0
                for prev in reversed(current):
                    if overlap_len + len(prev) <= self.chunk_overlap or not overlap_buf:
                        if overlap_len + len(prev) <= self.chunk_size - piece_len:
                            overlap_buf.insert(0, prev)
                            overlap_len += len(prev)
                        else:
                            break
                    else:
                        break
                current = overlap_buf
                current_len = overlap_len
            current.append(piece)
            current_len += piece_len

        if current:
            chunk_text = "".join(current).strip()
            if chunk_text:
                chunks.append(chunk_text)
        return chunks

    def split_text(self, text: str) -> List[str]:
        return self._recursive_split(text.strip(), self.separators)

    def split_documents(self, documents: Sequence[Document]) -> List[Document]:
        """批量切分 Document 列表，并自动注入 chunk_index、start_index 等溯源元数据。"""
        result: List[Document] = []
        for doc in documents:
            chunks = self.split_text(doc.page_content)
            cursor = 0
            for idx, chunk_text in enumerate(chunks):
                pos = doc.page_content.find(chunk_text[:20], max(0, cursor - self.chunk_overlap))
                if pos != -1:
                    cursor = pos
                meta = {
                    **doc.metadata,
                    "chunk_index": idx,
                    "total_chunks": len(chunks),
                    "start_index": pos if pos != -1 else 0,
                    "chunk_length": len(chunk_text),
                }
                result.append(Document(page_content=chunk_text, metadata=meta))
        return result


def run_chunking_experiment(sample_text: str) -> List[Dict[str, Any]]:
    """对比多组 chunk_size / chunk_overlap 参数的切片质量指标。"""
    configs = [
        (80, 15),
        (140, 25),
        (220, 40),
    ]
    report: List[Dict[str, Any]] = []
    for size, overlap in configs:
        splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
        chunks = splitter.split_text(sample_text)
        # 统计以中文句末标点结尾的切片比例（衡量断句完整度）
        clean_endings = sum(1 for c in chunks if c[-1] in "。！？；")
        total_chars = sum(len(c) for c in chunks)
        report.append({
            "chunk_size": size,
            "chunk_overlap": overlap,
            "num_chunks": len(chunks),
            "avg_len": round(total_chars / max(1, len(chunks)), 1),
            "clean_sentence_ratio": round(clean_endings / max(1, len(chunks)), 2),
            "overhead_ratio": round(total_chars / max(1, len(sample_text)), 2),
        })
    return report


if __name__ == "__main__":
    HANDBOOK = (
        "第一章 杭州星辰数科差旅报销制度。员工因公出差前须在OA系统提交出差申请，经直属主管审批通过后方可订票。"
        "一线城市（北京、上海、深圳、广州）酒店住宿标准为每晚不超过600元，新一线城市（杭州、南京、成都、武汉）每晚不超过480元。"
        "餐饮补助按每天100元定额发放，无需提供餐饮发票。\n\n"
        "第二章 研发代码发布规范。所有后端服务合并到 main 分支前，必须在 GitLab 发起 Merge Request，至少经过一位架构师或高级工程师 Review 通过。"
        "CI 流水线需保证 pytest 单元测试通过率达到100%，严禁在周五17:00之后向生产环境执行非紧急发布！"
    )
    print("=== 分块参数调优实验报告 ===")
    for row in run_chunking_experiment(HANDBOOK):
        print(row)

    docs = [Document(page_content=HANDBOOK, metadata={"source": "星辰员工手册.md", "dept": "综合"})]
    chunks = RecursiveCharacterTextSplitter(chunk_size=140, chunk_overlap=25).split_documents(docs)
    print(f"\n最佳配置切分预览（共 {len(chunks)} 块）：")
    for c in chunks:
        print(f"  [Chunk {c.metadata['chunk_index']}] len={c.metadata['chunk_length']} | {c.page_content}")


### 🔍 步骤 2 代码深度解析

1. `_split_with_separator` 把中文句号/感叹号保留在前一句末尾，避免了「下一块以句号开头」的丑陋切分。
2. `split_documents` 在切分同时保留了原文档所有 `metadata` 并追加 `chunk_index`、`total_chunks`、`start_index`，为后面做引用高亮提供精确字符坐标。
3. `run_chunking_experiment` 给出了非常实用的三项量化指标：切片数、完整句结尾比例（`clean_sentence_ratio`）和重叠冗余率（`overhead_ratio`）。

**预期运行输出参考：**
```text
=== 分块参数调优实验报告 ===
{'chunk_size': 80, 'chunk_overlap': 15, 'num_chunks': 5, 'avg_len': 66.2, 'clean_sentence_ratio': 1.0, 'overhead_ratio': 1.15}
{'chunk_size': 140, 'chunk_overlap': 25, 'num_chunks': 3, 'avg_len': 98.7, 'clean_sentence_ratio': 1.0, 'overhead_ratio': 1.03}
{'chunk_size': 220, 'chunk_overlap': 40, 'num_chunks': 2, 'avg_len': 144.0, 'clean_sentence_ratio': 1.0, 'overhead_ratio': 1.0}

最佳配置切分预览（共 3 块）：
  [Chunk 0] len=129 | 第一章 杭州星辰数科差旅报销制度。员工因公出差前须在OA系统提交出差申请，经直属主管审批通过后方可订票。一线城市（北京、上海、深圳、广州）酒店住宿标准为每晚不超过600元，新一线城市（杭州、南京、成都、武汉）每晚不超过480元。
  [Chunk 1] len=31 | 餐饮补助按每天100元定额发放，无需提供餐饮发票。
  [Chunk 2] len=136 | 第二章 研发代码发布规范。所有后端服务合并到 main 分支前，必须在 GitLab 发起 Merge Request，至少经过一位架构师或高级工程师 Review 通过。CI 流水线需保证 pytest 单元测试通过率达到100%，严禁在周五17:00之后向生产环境执行非紧急发布！
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day28/tests/test_day28.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 28 自动化单元测试：验证多格式文档加载器与中文递归分块器。"""

import sys
import zipfile
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import CSVLoader, DocxLoader, Document, HTMLLoader, MarkdownLoader, PagePDFLoader
from src.day28_text_splitter import RecursiveCharacterTextSplitter, run_chunking_experiment


def test_multi_format_loaders(tmp_path):
    # Markdown
    md = MarkdownLoader().load_text("# 年假\n满一年5天。\n## 病假\n需三甲医院证明。")
    assert len(md) == 2 and md[0].metadata["section"] == "年假" and md[1].metadata["section"] == "病假"

    # HTML 过滤脚本
    html = HTMLLoader().load_text("<html><style>.a{}</style><script>evil()</script><body><p>星辰公告</p></body></html>")
    assert html[0].page_content == "星辰公告"
    assert "evil" not in html[0].page_content

    # CSV 表头绑定
    csv_docs = CSVLoader().load_text("城市,上限\n杭州,480")
    assert "城市: 杭州" in csv_docs[0].page_content and "上限: 480" in csv_docs[0].page_content

    # 构造真实最小 .docx 文件测试 DocxLoader
    docx_path = tmp_path / "policy.docx"
    xml = (
        '<?xml version="1.0" encoding="UTF-8"?>'
        '<w:document xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main">'
        '<w:body><w:p><w:r><w:t>星辰数科保密协议第一条</w:t></w:r></w:p></w:body></w:document>'
    )
    with zipfile.ZipFile(docx_path, "w") as zf:
        zf.writestr("word/document.xml", xml)
    docx_docs = DocxLoader().load(docx_path)
    assert "星辰数科保密协议第一条" in docx_docs[0].page_content

    # PDF 分页
    pdf_docs = PagePDFLoader().load_pages(["P1内容", "P2内容"])
    assert len(pdf_docs) == 2 and pdf_docs[1].metadata["page"] == 2


def test_chinese_recursive_splitter_and_experiment():
    text = "第一句是关于报销的详细规则说明。第二句说明一线城市住宿标准600元。第三句说明餐补每天100元。第四句说明审批流程需要主管确认。"
    splitter = RecursiveCharacterTextSplitter(chunk_size=45, chunk_overlap=10)
    docs = splitter.split_documents([Document(page_content=text, metadata={"source": "rule.md"})])
    assert len(docs) >= 3
    for idx, d in enumerate(docs):
        assert len(d.page_content) <= 45
        assert d.metadata["source"] == "rule.md"
        assert d.metadata["chunk_index"] == idx

    report = run_chunking_experiment(text * 2)
    assert len(report) == 3
    assert all(r["num_chunks"] >= 1 for r in report)

    with pytest.raises(ValueError):
        RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=50)

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：作业一：实现 Markdown 标题上下文注入器（Context-Enriched Splitter）

**题目要求：**
很多文档的切片脱离标题后语义模糊（例如切片内容只有「需提前3天申请」，不知道是年假还是出差）。请编写函数 `enrich_chunks_with_headers(chunks)`，将 `metadata['section']` 和 `metadata['source']` 自动拼接到每个切片正文的开头，格式为 `【来源：{source} | 章节：{section}】\n{page_content}`。

**思路提示：** 遍历 `chunks`，返回新的 `Document` 列表（不破坏原对象），避免重复注入。


In [ ]:
"""Day 28 作业一参考答案：切片标题上下文自动注入器"""
import sys
from pathlib import Path
from typing import List

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document, MarkdownLoader


def enrich_chunks_with_headers(chunks: List[Document]) -> List[Document]:
    enriched: List[Document] = []
    for doc in chunks:
        source = doc.metadata.get("source", "未知文档")
        section = doc.metadata.get("section", "正文")
        prefix = f"【来源：{source} | 章节：{section}】\n"
        content = doc.page_content if doc.page_content.startswith("【来源：") else prefix + doc.page_content
        enriched.append(Document(page_content=content, metadata=dict(doc.metadata)))
    return enriched


if __name__ == "__main__":
    raw = MarkdownLoader().load_text("# 年假管理制度\n需提前3个工作日提交OA申请。", source="HR手册.md")
    result = enrich_chunks_with_headers(raw)
    print(result[0].page_content)
    assert "年假管理制度" in result[0].page_content


**答案解析与评分标准：**

这是企业级 RAG 中性价比最高的提分技巧（Anthropic 称为 Contextual Retrieval 的轻量版），几行代码就能大幅提升孤立切片的检索召回率。


### 作业 2（企业实战）：作业二：实现基于内容哈希的文档切片去重过滤器

**题目要求：**
企业内网经常存在多份重复上传的制度附件（如「报销制度_v1.md」和「报销制度_副本.md」）。请实现 `deduplicate_chunks(chunks)`：对文本去除空白后计算 SHA-256 哈希，剔除完全重复的切片，返回 `(unique_chunks, duplicate_count)`。

**思路提示：** 用 `hashlib.sha256(re.sub(r'\s+', '', doc.page_content).encode()).hexdigest()` 作为去重指纹。


In [ ]:
"""Day 28 作业二参考答案：基于 SHA-256 的切片去重过滤器"""
import hashlib
import re
import sys
from pathlib import Path
from typing import List, Set, Tuple

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document


def deduplicate_chunks(chunks: List[Document]) -> Tuple[List[Document], int]:
    seen: Set[str] = set()
    unique: List[Document] = []
    dup_count = 0
    for doc in chunks:
        normalized = re.sub(r"\s+", "", doc.page_content)
        digest = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
        if digest in seen:
            dup_count += 1
            continue
        seen.add(digest)
        doc_copy = Document(page_content=doc.page_content, metadata={**doc.metadata, "content_hash": digest[:12]})
        unique.append(doc_copy)
    return unique, dup_count


if __name__ == "__main__":
    docs = [
        Document("一线城市住宿标准600元。", {"source": "a.md"}),
        Document("一线城市住宿标准 600元。\n", {"source": "a_copy.md"}),
        Document("餐补每天100元。", {"source": "b.md"}),
    ]
    uniq, dups = deduplicate_chunks(docs)
    print(f"去重前 {len(docs)} -> 去重后 {len(uniq)}，剔除重复 {dups} 条")
    assert len(uniq) == 2 and dups == 1


**答案解析与评分标准：**

如果不去重，向量库 Top-3 检索结果很可能返回 3 份一模一样的副本，直接挤占掉其他真正有用的参考资料。


### 作业 3（架构进阶）：作业三：实现保护代码块的 Markdown 智能分块器

**题目要求：**
研发知识库里有大量 ```` ```python ... ``` ```` 代码块，普通分块器容易把代码从中间拦腰切断。请编写 `CodeAwareMarkdownSplitter`，确保任何被三重反引号包裹的代码块都作为一个不可分割的原子块保留完整。

**思路提示：** 先用正则 `re.split(r'(```.*?```)', text, flags=re.DOTALL)` 把文本拆成「普通文本段」和「代码块段」，普通段走递归分块，代码块段保持整体不切分。


In [ ]:
"""Day 28 作业三参考答案：保护代码块完整性的 Markdown 分块器"""
import re
import sys
from pathlib import Path
from typing import List

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_text_splitter import RecursiveCharacterTextSplitter


class CodeAwareMarkdownSplitter:
    CODE_BLOCK_RE = re.compile(r"(```[\s\S]*?```)")

    def __init__(self, chunk_size: int = 120, chunk_overlap: int = 20) -> None:
        self._text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )

    def split_text(self, md_text: str) -> List[str]:
        segments = self.CODE_BLOCK_RE.split(md_text)
        chunks: List[str] = []
        for seg in segments:
            seg = seg.strip()
            if not seg:
                continue
            if seg.startswith("```") and seg.endswith("```"):
                # 代码块整体保留，绝不拦腰切断
                chunks.append(seg)
            else:
                chunks.extend(self._text_splitter.split_text(seg))
        return chunks


if __name__ == "__main__":
    md = (
        "星辰后端要求所有接口必须使用 Pydantic 校验。\n"
        "```python\nclass QueryReq(BaseModel):\n    question: str\n    top_k: int = 5\n```\n"
        "发布前必须通过全部单元测试。"
    )
    splitter = CodeAwareMarkdownSplitter(chunk_size=30, chunk_overlap=5)
    result = splitter.split_text(md)
    for i, c in enumerate(result):
        print(f"[{i}] {c!r}")
    assert any("class QueryReq" in c and c.endswith("```") for c in result)


**答案解析与评分标准：**

研发团队内部知识库（API 文档、故障排查手册）如果把代码块切烂了，RAG 生成的代码 100% 跑不通。按内容类型分治切分是企业文档解析的进阶必修课。


---

## 🎯 老陈收尾复盘

今天这一步地基打得非常扎实！记住老陈的中文分块黄金法则：分隔符优先级一定要按 `['\n\n', '\n', '。', '！', '？', '；', '，', ' ', '']` 逐级降级，并且务必保留 10%~20% 的 `chunk_overlap` 以及完整的 `metadata`（来源文件、章节标题、页码、字符偏移量），这样后面做引用溯源高亮时你才会感激今天的自己！明天咱们进军向量数据库。
